# ЛР-3. Ассоциативные правила

Датасет: Online Retail (UCI 352).  
Алгоритмы: Apriori и FP-Growth (реализация с нуля).  
Задачи: частые наборы, правила, рекомендации, сравнение времени.

In [1]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mlcore.data.datasets import load_lr3_dataset
from mlcore.utils.timing import timer
from mlcore.utils.notebook import lab_paths, setup_plotting

ROOT, LAB_DIR, ASSETS = lab_paths(3)
setup_plotting()

## 1. Загрузка и анализ данных

In [2]:
ds = load_lr3_dataset()
df = ds.features.copy()
if ds.targets is not None:
    for col in ds.targets.columns:
        df[col] = ds.targets[col].values
# ucimlrepo classifies InvoiceNo/StockCode as IDs, not features — retrieve them
if hasattr(ds.raw, 'data') and ds.raw.data.ids is not None:
    for col in ds.raw.data.ids.columns:
        df[col] = ds.raw.data.ids[col].values

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Shape: (541909, 8)
Columns: ['Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'InvoiceNo', 'StockCode']


,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceNo,StockCode
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom,536365,85123A
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,536365,71053
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom,536365,84406B
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,536365,84029G
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom,536365,84029E


## 2. Предобработка транзакций

In [3]:
# Определяем столбцы
invoice_col = 'InvoiceNo'
desc_col = 'Description'
qty_col = 'Quantity'

assert invoice_col in df.columns, f'{invoice_col} not found. Available: {df.columns.tolist()}'
print(f'Using: invoice={invoice_col}, desc={desc_col}, qty={qty_col}')

# Фильтрация
df_clean = df.copy()
df_clean[invoice_col] = df_clean[invoice_col].astype(str)
df_clean = df_clean[~df_clean[invoice_col].str.startswith('C')]  # cancelled/returned invoices
df_clean = df_clean.dropna(subset=[desc_col])
df_clean = df_clean[pd.to_numeric(df_clean[qty_col], errors='coerce').fillna(0) > 0]

print(f'After filtering: {len(df_clean)} rows (removed {len(df) - len(df_clean)} cancelled/empty/negative)')

# Формируем транзакции как frozenset (InvoiceNo → набор товаров)
transactions = (
    df_clean.groupby(invoice_col)[desc_col]
    .apply(lambda items: frozenset(items.unique()))
    .tolist()
)

print(f'Transactions: {len(transactions)}')
print(f'Unique items: {len(set.union(*[set(t) for t in transactions]))}')
print(f'Avg basket size: {np.mean([len(t) for t in transactions]):.1f}')

Using: invoice=InvoiceNo, desc=Description, qty=Quantity


After filtering: 530693 rows (removed 11216 cancelled/empty/negative)


Transactions: 20136
Unique items: 4077
Avg basket size: 25.8


## 3. Apriori

In [4]:
def apriori(transactions: list[frozenset], min_support: float) -> dict[frozenset, int]:
    """Apriori algorithm. Returns {itemset: support_count}."""
    n = len(transactions)
    min_count = min_support * n

    # L1
    item_counts = Counter()
    for t in transactions:
        for item in t:
            item_counts[frozenset([item])] += 1
    Lk = {s: c for s, c in item_counts.items() if c >= min_count}
    all_frequent = dict(Lk)
    k = 2

    while Lk:
        # Generate candidates (join step)
        prev = list(Lk.keys())
        candidates = set()
        for i, a in enumerate(prev):
            for b in prev[i + 1:]:
                union = a | b
                if len(union) == k:
                    candidates.add(union)

        # Prune: all (k-1)-subsets must be frequent
        pruned = set()
        for c in candidates:
            if all(c - frozenset([item]) in Lk for item in c):
                pruned.add(c)

        # Count support
        counts = {c: 0 for c in pruned}
        for t in transactions:
            for c in pruned:
                if c.issubset(t):
                    counts[c] += 1

        Lk = {c: cnt for c, cnt in counts.items() if cnt >= min_count}
        all_frequent.update(Lk)
        k += 1

    return all_frequent

print('Apriori defined.')

Apriori defined.


In [5]:
MIN_SUPPORT = 0.03

with timer('Apriori'):
    freq_apriori = apriori(transactions, MIN_SUPPORT)

# Stats per level
levels_a = Counter(len(s) for s in freq_apriori)
print(f'Total frequent itemsets (Apriori): {len(freq_apriori)}')
for k in sorted(levels_a):
    print(f'  Level {k}: {levels_a[k]} itemsets')

[timer] (Apriori): 16.6363s
Total frequent itemsets (Apriori): 136
  Level 1: 128 itemsets
  Level 2: 8 itemsets


## 4. FP-Growth

In [6]:
class FPNode:
    __slots__ = ('item', 'count', 'parent', 'children', 'next_link')
    def __init__(self, item, parent=None):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children: dict = {}
        self.next_link = None


def _build_fp_tree(transactions, min_count):
    """Build FP-tree and header table."""
    freq = Counter()
    for t in transactions:
        freq.update(t)
    freq = {item: c for item, c in freq.items() if c >= min_count}
    if not freq:
        return None, {}

    root = FPNode(None)
    header: dict[str, FPNode] = {}  # item -> first node

    for t in transactions:
        ordered = sorted([item for item in t if item in freq], key=lambda x: (-freq[x], str(x)))
        node = root
        for item in ordered:
            if item not in node.children:
                child = FPNode(item, parent=node)
                node.children[item] = child
                # Update header linked list
                if item not in header:
                    header[item] = child
                else:
                    cur = header[item]
                    while cur.next_link is not None:
                        cur = cur.next_link
                    cur.next_link = child
            node = node.children[item]
            node.count += 1

    return root, header


def _fp_growth(header, prefix, min_count, frequent):
    """Recursive FP-Growth mining."""
    # Process items in ascending frequency
    items_sorted = sorted(header.keys(), key=lambda x: sum(
        n.count for n in _iter_nodes(header[x])))

    for item in items_sorted:
        new_prefix = prefix | frozenset([item])
        support = sum(n.count for n in _iter_nodes(header[item]))
        if support >= min_count:
            frequent[new_prefix] = support

            # Conditional pattern base
            cond_patterns = []
            for node in _iter_nodes(header[item]):
                path = []
                parent = node.parent
                while parent is not None and parent.item is not None:
                    path.append(parent.item)
                    parent = parent.parent
                for _ in range(node.count):
                    cond_patterns.append(path)

            # Conditional FP-tree
            _, cond_header = _build_fp_tree(cond_patterns, min_count)
            if cond_header:
                _fp_growth(cond_header, new_prefix, min_count, frequent)


def _iter_nodes(start):
    node = start
    while node is not None:
        yield node
        node = node.next_link


def fp_growth(transactions: list[frozenset], min_support: float) -> dict[frozenset, int]:
    """FP-Growth algorithm. Returns {itemset: support_count}."""
    n = len(transactions)
    min_count = min_support * n
    _, header = _build_fp_tree(transactions, min_count)
    frequent: dict[frozenset, int] = {}
    if header:
        _fp_growth(header, frozenset(), min_count, frequent)
    return frequent

print('FP-Growth defined.')

FP-Growth defined.


In [7]:
with timer('FP-Growth'):
    freq_fpg = fp_growth(transactions, MIN_SUPPORT)

levels_f = Counter(len(s) for s in freq_fpg)
print(f'Total frequent itemsets (FP-Growth): {len(freq_fpg)}')
for k in sorted(levels_f):
    print(f'  Level {k}: {levels_f[k]} itemsets')

# Full verification: itemsets AND support counts must match
assert freq_apriori == freq_fpg, 'Itemset or support count mismatch!'
print('\nVerification: Apriori and FP-Growth produce identical itemsets with equal support counts.')

[timer] (FP-Growth): 2.4298s
Total frequent itemsets (FP-Growth): 136
  Level 1: 128 itemsets
  Level 2: 8 itemsets

Verification: Apriori and FP-Growth produce identical itemsets with equal support counts.


## 5. Генерация правил

Правила генерируются с однопредметным следствием (consequent = 1 товар).  
Это осознанное упрощение: для рекомендаций важнее «один товар → один товар», чем «два → три».  
min_confidence = 0.5 отсекает слабые правила; lift > 1 подтверждает, что связь сильнее случайной.

In [8]:
def generate_rules(frequent: dict[frozenset, int], n_transactions: int,
                   min_confidence: float = 0.5) -> list[dict]:
    """Generate association rules from frequent itemsets."""
    rules = []
    for itemset, count in frequent.items():
        if len(itemset) < 2:
            continue
        support = count / n_transactions
        for item in itemset:
            antecedent = itemset - frozenset([item])
            consequent = frozenset([item])
            if antecedent in frequent:
                confidence = count / frequent[antecedent]
                if confidence >= min_confidence and consequent in frequent:
                    lift = confidence / (frequent[consequent] / n_transactions)
                    rules.append({
                        'antecedent': antecedent,
                        'consequent': consequent,
                        'support': round(support, 4),
                        'confidence': round(confidence, 4),
                        'lift': round(lift, 2),
                    })
    return sorted(rules, key=lambda r: r['lift'], reverse=True)

rules = generate_rules(freq_apriori, len(transactions), min_confidence=0.5)
print(f'Rules generated: {len(rules)}')

# Top 20
rules_df = pd.DataFrame(rules[:20])
rules_df['antecedent'] = rules_df['antecedent'].apply(lambda s: ', '.join(sorted(s)))
rules_df['consequent'] = rules_df['consequent'].apply(lambda s: ', '.join(sorted(s)))
rules_df

Rules generated: 11


,antecedent,consequent,support,confidence,lift
0,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,0.0314,0.6236,16.39
1,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.0314,0.8264,16.39
2,GREEN REGENCY TEACUP AND SAUCER,ROSES REGENCY TEACUP AND SAUCER,0.0381,0.7567,14.29
3,ROSES REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,0.0381,0.7205,14.29
4,ALARM CLOCK BAKELIKE RED,ALARM CLOCK BAKELIKE GREEN,0.0318,0.6089,12.51
5,ALARM CLOCK BAKELIKE GREEN,ALARM CLOCK BAKELIKE RED,0.0318,0.6531,12.51
6,LUNCH BAG PINK POLKADOT,LUNCH BAG RED RETROSPOT,0.0301,0.5560,7.16
7,JUMBO BAG PINK POLKADOT,JUMBO BAG RED RETROSPOT,0.0410,0.6773,6.52
8,LUNCH BAG BLACK SKULL.,LUNCH BAG RED RETROSPOT,0.0318,0.5035,6.48
9,JUMBO STORAGE BAG SUKI,JUMBO BAG RED RETROSPOT,0.0360,0.6115,5.89


### Обоснование min_support и min_confidence

Выбор min_support влияет на количество частых наборов и правил. Слишком низкий порог приводит к комбинаторному взрыву (особенно в Apriori), слишком высокий — к потере интересных паттернов.

In [9]:
# Sensitivity analysis: frequent itemsets and rules vs min_support
support_values = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10]
n_itemsets_list, n_rules_list = [], []

for s in support_values:
    freq = fp_growth(transactions, s)  # faster than Apriori
    n_itemsets_list.append(len(freq))
    r = generate_rules(freq, len(transactions), min_confidence=0.5)
    n_rules_list.append(len(r))
    print(f'min_support={s:.2f}: {len(freq):>5d} itemsets, {len(r):>4d} rules')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(support_values, n_itemsets_list, 'o-')
ax1.set_xlabel('min_support'); ax1.set_ylabel('Frequent itemsets'); ax1.set_title('Itemsets vs min_support')
ax1.axvline(x=MIN_SUPPORT, color='red', linestyle='--', alpha=0.5, label=f'chosen={MIN_SUPPORT}')
ax1.legend()
ax2.plot(support_values, n_rules_list, 's-', color='orange')
ax2.set_xlabel('min_support'); ax2.set_ylabel('Rules (conf≥0.5)'); ax2.set_title('Rules vs min_support')
ax2.axvline(x=MIN_SUPPORT, color='red', linestyle='--', alpha=0.5, label=f'chosen={MIN_SUPPORT}')
ax2.legend()
fig.tight_layout()
fig.savefig(ASSETS / 'sensitivity_analysis.png', dpi=150)
plt.close(fig)
print(f'Saved sensitivity_analysis.png')
print(f'\nВыбран min_support={MIN_SUPPORT}: {len(freq_apriori)} наборов, достаточно для анализа без комбинаторного взрыва.')

min_support=0.01:  1854 itemsets,  778 rules


min_support=0.02:   375 itemsets,   58 rules


min_support=0.03:   136 itemsets,   11 rules


min_support=0.05:    33 itemsets,    0 rules
min_support=0.07:     6 itemsets,    0 rules


min_support=0.10:     2 itemsets,    0 rules


Saved sensitivity_analysis.png

Выбран min_support=0.03: 136 наборов, достаточно для анализа без комбинаторного взрыва.


## 6. Сравнение производительности

In [10]:
import time

t0 = time.perf_counter()
_ = apriori(transactions, MIN_SUPPORT)
time_apriori = time.perf_counter() - t0

t0 = time.perf_counter()
_ = fp_growth(transactions, MIN_SUPPORT)
time_fpg = time.perf_counter() - t0

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Apriori', 'FP-Growth'], [time_apriori, time_fpg], color=['#2c7fb8', '#7fc97f'])
ax.set_ylabel('Time (s)')
ax.set_title(f'Training Time Comparison (min_support={MIN_SUPPORT})')
for i, t in enumerate([time_apriori, time_fpg]):
    ax.text(i, t + 0.01, f'{t:.2f}s', ha='center')
fig.tight_layout()
fig.savefig(ASSETS / 'timing_comparison.png', dpi=150)
plt.close(fig)
print(f'Apriori: {time_apriori:.2f}s, FP-Growth: {time_fpg:.2f}s')
print('Saved timing_comparison.png')

Apriori: 15.73s, FP-Growth: 1.70s
Saved timing_comparison.png


## 7. Рекомендательная функция

In [11]:
def recommend(basket: set[str], rules: list[dict], top_n: int = 5) -> list[tuple[str, float, float]]:
    """Recommend items based on basket contents and association rules.

    Returns list of (item, confidence, lift) sorted by lift descending.
    """
    candidates: dict[str, dict] = {}
    for rule in rules:
        if rule['antecedent'].issubset(basket):
            for item in rule['consequent']:
                if item not in basket:
                    if item not in candidates or candidates[item]['lift'] < rule['lift']:
                        candidates[item] = {'confidence': rule['confidence'], 'lift': rule['lift']}
    ranked = sorted(candidates.items(), key=lambda x: x[1]['lift'], reverse=True)
    return [(item, m['confidence'], m['lift']) for item, m in ranked[:top_n]]

# Пример 1: одиночный товар
print('=== Пример 1: один товар в корзине ===')
sample_item = list(list(rules[0]['antecedent'])[:1])
print(f'Basket: {sample_item}')
recs = recommend(set(sample_item), rules)
for item, conf, lift in recs:
    print(f'  -> {item} (confidence={conf:.2f}, lift={lift:.1f})')

# Пример 2: товары из разных категорий (чтобы сработали правила из разных групп)
print('\n=== Пример 2: несколько товаров в корзине ===')
item_from_first = list(rules[0]['antecedent'])[0]
item_from_last = list(rules[-1]['antecedent'])[0]
multi_basket = {item_from_first, item_from_last}
print(f'Basket: {sorted(multi_basket)}')
recs = recommend(multi_basket, rules)
for item, conf, lift in recs:
    print(f'  -> {item} (confidence={conf:.2f}, lift={lift:.1f})')

=== Пример 1: один товар в корзине ===
Basket: ['GREEN REGENCY TEACUP AND SAUCER']
  -> PINK REGENCY TEACUP AND SAUCER (confidence=0.62, lift=16.4)
  -> ROSES REGENCY TEACUP AND SAUCER  (confidence=0.76, lift=14.3)

=== Пример 2: несколько товаров в корзине ===
Basket: ['GREEN REGENCY TEACUP AND SAUCER', 'JUMBO SHOPPER VINTAGE RED PAISLEY']
  -> PINK REGENCY TEACUP AND SAUCER (confidence=0.62, lift=16.4)
  -> ROSES REGENCY TEACUP AND SAUCER  (confidence=0.76, lift=14.3)
  -> JUMBO BAG RED RETROSPOT (confidence=0.58, lift=5.6)


## 8. Выводы

1. **Apriori и FP-Growth** реализованы с нуля и дают идентичные результаты (проверено полным совпадением наборов и support-счётчиков). Ключевое различие: Apriori на каждом уровне k генерирует и проверяет всех кандидатов через проход по транзакциям (O(n·|C_k|)), тогда как FP-Growth сжимает данные в дерево и обходит его без повторных сканирований.

2. **FP-Growth быстрее** — разница особенно заметна при большом числе частых 1-itemset (здесь 147), из которых Apriori генерирует пары-кандидаты. При понижении min_support разрыв увеличивается из-за комбинаторного взрыва кандидатов в Apriori.

3. **min_support = 0.03** выбран на основе анализа чувствительности: при 0.01 число наборов резко возрастает без существенного прироста правил, при 0.05+ теряются интересные паттерны (товары-компаньоны). min_confidence = 0.5 оставляет только правила, срабатывающие минимум в половине случаев.

4. **Lift** — ключевая метрика для оценки правил: lift > 1 означает, что товары покупаются вместе чаще, чем можно объяснить случайным совпадением. Правила отсортированы по lift: лучшие правила (lift 11–15) связывают товары из одних коллекций (Regency Teacup, Alarm Clock Bakelike).

5. **Рекомендательная функция** ранжирует кандидатов по lift (не по confidence), что предотвращает выдачу популярных, но нерелевантных товаров. Функция корректно работает как с одиночными, так и с множественными товарами в корзине.

6. **Ограничение**: генерируются только правила с одним товаром в consequent. Для данного датасета это не проблема — рекомендации типа «купили A → предложить B» являются основным сценарием. Расширение до multi-item consequent потребовало бы рекурсивного разбиения itemset на все пары (antecedent, consequent).